# RQ2: Are there significant differences in developers’ perceptions of security practices across groups with different experience levels, professional roles, and organizational types?

For each practice and attribute, the groups Direct vs. Indirect experience, Large vs. Small organization and Specialist vs. Non-specialist are compared with two-sided Mann-Whitney U tests, with Cliff's delta as the effect size and Holm-Bonferroni adjusted p-values (one family of 16 tests, one per practice, for each comparison and attribute). 'Não Aplicável' (Not Applicable) answers are excluded.

Outputs (`results/RQ2/`):

- `analysis_results_raw.xlsx`: means, p-values, Holm adjusted p-values and effect sizes;
- `analysis_results_formatted.xlsx`: the same results in a formatted table.

It also saves `results/tools_usage_frequency_responses.xlsx`: the answers about the frequency of use of each category of automated security tools.

The LaTeX table is generated from `analysis_results_raw.xlsx` by `scripts/generate_latex_tables.py`.

## Pre-processing

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from cliffs_delta import cliffs_delta

In [2]:
data_path = Path('data')

df_o = pd.read_excel(data_path / 'qualitative_responses_ptbr.xlsx')

df = df_o[:]

df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace('\n', '', regex=True)

print(len(df))

print(df.columns.tolist())

95
['ID', 'Carimbo de data/hora', 'Após ter recebido esclarecimentos sobre a natureza da pesquisa, seus objetivos e métodos, eu declaro que sou maior de 18 anos e aceito participar por meio do formulário de participação, conforme as disposições da Lei Geral de Proteção de Dados (Lei nº 13.709/2018).Em caso de resposta negativa, o formulário será encerrado, garantindo assim o respeito à autonomia e privacidade dos participantes, conforme estabelecido pela legislação vigente.', 'Qual é a sua idade?', 'Em qual estado do Brasil você mora atualmente?', 'Qual é o seu gênero?', 'Qual é o seu nível de educação formal concluída?', 'Qual seu modelo de trabalho?', 'Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?', 'Por favor, indique a senioridade da posição que ocupa.', 'Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?', 'Quantos anos de experiência você tem em funções relacionadas a Segurança de 

In [3]:
print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())


df = df[~(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?']
          == 'Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software')]

print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())
print(df.shape)

<StringArray>
[                            'Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software',
 'Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.',
                                           'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
Length: 3, dtype: str
<StringArray>
['Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.', 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
Length: 2, dtype: str
(83, 103)


Analyses

In [4]:
categories = {
    "3rd Party Vulnerability Scanning": [
        "Dependabot", "PHP Security Checker", "Snyk", "Clair", "Blackduck",
        "Retire.JS", "SRC:CLR", "Anchore", "Node Security Platform", "Trivy",
        "Sonatype", "Whitesource"
    ],
    "Static Application Security Testing": [  # SAST
        "CodeQL", "CheckStyle", "Coverity", "SonarQube", "Bandit", "PHPStan",
        "LGTM", "Brakeman", "ESLint Security", "Dart Analyzer", "DevSkim"
    ],
    "Dynamic Application Security Testing": [  # DAST
        "Qark", "Checkmarx", "CFN NAG"
    ],
    "Secure Configuration/Hardening": [
        "Chef", "Ansible", "Terraform", "Puppet", "Phan", "Seccomp",
        "CloudCustodian", "Turbot"
    ],
    "Compliance/Hardening Checks": [
        "Bane", "Ansible-Lint", "Blackbox", "Puppet-Lint", "InSpec",
        "Kube Audit", "Foodcritic", "Cookstyle", "conftest"
    ],
    "Secrets Management": [
        "Pinterest Knox", "GitLeaks", "CyberArk Conjur",
        "ThoughtWorks Talisman", "Berglas", "Docker Secrets", "Trufflehog"
    ]
}

In [5]:
tool_cols = [
    col for col in df.columns
    if "categorias de ferramentas" in str(col).lower()
]

print(f"\n{len(tool_cols)} columns found for the question about tools:")
for c in tool_cols:
    print("-", c)

tools_df = df[tool_cols].copy()


6 columns found for the question about tools:
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [3rd Party Vulnerability Scanning Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Static Application Security Testing Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Dynamic Application Security Testing Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não 

In [6]:
tools_df = tools_df.map(
    lambda x: x.strip() if isinstance(x, str) else x
)

In [7]:
tools_df.insert(0, "Respondent", range(1, len(tools_df) + 1))

In [8]:
likert_scale = [
    "Diariamente",
    "Semanalmente",
    "Mensalmente",
    "Anualmente",
    "Não Aplicável",
]

In [9]:
result = {}

for category in tools_df.columns[1:]:  # ignores the "Respondent" column
    result[category] = {}
    for option in likert_scale:
        # Selects all rows where the respondent chose this option
        participants = tools_df[tools_df[category] == option][[
            "Respondent", category]]

        # Stores the individual responses
        result[category][option] = participants


In [10]:
results_path = Path("results/RQ2")
results_path.mkdir(parents=True, exist_ok=True)

# Answers about the use of the tool categories (not an RQ2 result): saved directly in results/
with pd.ExcelWriter(results_path.parent / "tools_usage_frequency_responses.xlsx") as writer:
    for i, (category, responses_by_option) in enumerate(result.items()):
        frames = []
        for option, responses in responses_by_option.items():
            temp = responses.copy()
            temp["Option"] = option
            frames.append(temp)
        category_df = pd.concat(frames, ignore_index=True)
        # ensures uniqueness and respects the 31-character limit
        sheet_name = f"{category[:27]}_{i}"
        category_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\nSaved File: {results_path.parent}/tools_usage_frequency_responses.xlsx")



Saved File: results/tools_usage_frequency_responses.xlsx


## 'Not applicable' category


In [11]:
# List of survey columns related to security practices
security_practices_survey_cols = [
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Revisão de Código]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Modelagem de Ameaças]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Identificação de Casos de Abuso/Mau Uso]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Análise de Riscos Arquiteturais]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Penetração]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Riscos]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Vulnerabilidades]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Segurança Manuais/Automatizados]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Execução de Testes Fuzz]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Engenharia de Requisitos de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Codificação Segura]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Resposta Emergencial ao Código]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Configurações e Gerenciamento de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Monitoramento de Eventos de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Auditorias de Segurança Externas]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Segurança da Cadeia de Suprimentos de Software]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Revisão de Código]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Modelagem de Ameaças]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Identificação de Casos de Abuso/Mau Uso]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Análise de Riscos Arquiteturais]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Penetração]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Riscos]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Vulnerabilidades]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Segurança Manuais/Automatizados]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Execução de Testes Fuzz]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Engenharia de Requisitos de Segurança]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Codificação Segura]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Resposta Emergencial ao Código]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Configurações e Gerenciamento de Segurança]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Monitoramento de Eventos de Segurança]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Auditorias de Segurança Externas]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Segurança da Cadeia de Suprimentos de Software]'
]

In [12]:
not_applicable_summary = []

for column in security_practices_survey_cols:
    # Count occurrences of 'Não Aplicável' (Not Applicable)
    not_applicable_count = df[column].value_counts().get('Não Aplicável', 0)

    # Count total valid responses (excluding NaN for percentage calculation)
    total_valid_responses = df[column].dropna().shape[0]

    if total_valid_responses > 0:
        not_applicable_percentage = (
            not_applicable_count / total_valid_responses) * 100
    else:
        not_applicable_percentage = 0

    not_applicable_summary.append({
        'Column': column,
        'Not Applicable count': not_applicable_count,
        'Not Applicable percentage (%)': round(not_applicable_percentage, 2)
    })

df_not_applicable_summary = pd.DataFrame(not_applicable_summary)

print("DataFrame with the summary of 'Not Applicable' by column:")
display(df_not_applicable_summary)


DataFrame with the summary of 'Not Applicable' by column:


,Column,Not Applicable count,Not Applicable percentage (%)
0,"Com base em sua experiência, como você avaliar...",11,13.25
1,"Com base em sua experiência, como você avaliar...",20,24.10
2,"Com base em sua experiência, como você avaliar...",18,21.69
3,"Com base em sua experiência, como você avaliar...",14,16.87
4,"Com base em sua experiência, como você avaliar...",15,18.07
...,...,...,...
59,Qual a sua opinião quanto a frase a seguir apl...,0,0.00
60,Qual a sua opinião quanto a frase a seguir apl...,0,0.00
61,Qual a sua opinião quanto a frase a seguir apl...,0,0.00
62,Qual a sua opinião quanto a frase a seguir apl...,0,0.00


In [13]:
# Identify 'ease of use' and 'effectiveness' columns
ease_of_use_cols = [col for col in df_not_applicable_summary['Column']
                       if 'facilidade de uso' in col.lower()]
effectiveness_cols = [col for col in df_not_applicable_summary['Column']
                          if 'efetividade das seguintes práticas' in col.lower()]

# Filter the summary DataFrame to get the percentages
df_ease_of_use = df_not_applicable_summary[df_not_applicable_summary['Column'].isin(
    ease_of_use_cols)]
df_effectiveness = df_not_applicable_summary[df_not_applicable_summary['Column'].isin(
    effectiveness_cols)]

# Calculate the mean percentages for each group
ease_of_use_mean = df_ease_of_use['Not Applicable percentage (%)'].mean()
effectiveness_mean = df_effectiveness['Not Applicable percentage (%)'].mean()

# Create a new DataFrame with the results
df_applicability_summary = pd.DataFrame({
    'Attribute': ['Ease of use', 'Effectiveness'],
    'Mean Not Applicable percentage (%)': [round(ease_of_use_mean, 2), round(effectiveness_mean, 2)]
})

print("DataFrame summarizing the average of 'Not Applicable' by attribute:")
display(df_applicability_summary)

DataFrame summarizing the average of 'Not Applicable' by attribute:


,Attribute,Mean Not Applicable percentage (%)
0,Ease of use,22.74
1,Effectiveness,19.35


In [14]:
from pathlib import Path
data_path = Path('data')

df_o = pd.read_excel(
    data_path / 'survey_all_responses_ptbr.xlsx')

df = df_o[:]

df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace('\n', '', regex=True)

In [15]:
print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())


df = df[~(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?']
          == 'Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software')]

print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())
print(df.shape)

<StringArray>
[                            'Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software',
 'Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.',
                                           'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
Length: 3, dtype: str
<StringArray>
['Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.', 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
Length: 2, dtype: str
(83, 102)


In [16]:
print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].isna().sum())

df = df.drop(df[df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].isna()].index, axis=0)

print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].isna().sum())

0
0


In [17]:
practices = ['[Revisão de Código]',
            '[Modelagem de Ameaças]',
            '[Identificação de Casos de Abuso/Mau Uso]',
            '[Análise de Riscos Arquiteturais]',
            '[Testes de Penetração]',
            '[Gerenciamento de Riscos]',
            '[Gerenciamento de Vulnerabilidades]',
            '[Testes de Segurança Manuais/Automatizados]',
            '[Execução de Testes Fuzz]',
            '[Engenharia de Requisitos de Segurança]',
            '[Codificação Segura]',
            '[Resposta Emergencial ao Código]',
            '[Configurações e Gerenciamento de Segurança]',
            '[Monitoramento de Eventos de Segurança]',
            '[Auditorias de Segurança Externas]',
            '[Segurança da Cadeia de Suprimentos de Software]']

# English names of the practices, in the same order (used in the outputs)
practices_en = ['Code Review',
                'Threat Modeling',
                'Identifying Abuse/Misuse Cases',
                'Architectural Risk Analysis',
                'Penetration Testing',
                'Risk Management',
                'Vulnerability Management',
                'Manual/Automated Security Testing',
                'Performing Fuzz Testing',
                'Security Requirements Engineering',
                'Secure Coding',
                'Emergency Codebase Response',
                'Security Configuration and Management',
                'Monitoring for Security Events',
                'Undergoing External Security Audits',
                'Software Supply Chain Security']

# Portuguese name (as in the survey columns) -> English name
practice_name_en = {
    practice.strip('[]'): practice_en for practice, practice_en in zip(practices, practices_en)
}


In [18]:
frequency_cols = ['Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]'
                      ]

In [19]:
ease_of_use_cols = ['Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Revisão de Código]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Modelagem de Ameaças]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Identificação de Casos de Abuso/Mau Uso]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Análise de Riscos Arquiteturais]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Penetração]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Riscos]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Vulnerabilidades]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Segurança Manuais/Automatizados]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Execução de Testes Fuzz]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Engenharia de Requisitos de Segurança]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Codificação Segura]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Resposta Emergencial ao Código]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Configurações e Gerenciamento de Segurança]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Monitoramento de Eventos de Segurança]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Auditorias de Segurança Externas]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Segurança da Cadeia de Suprimentos de Software]'
                      ]

In [20]:
effectiveness_cols = ['Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]'
                       ]

In [21]:
essentiality_cols = ['Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Revisão de Código]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Modelagem de Ameaças]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Identificação de Casos de Abuso/Mau Uso]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Análise de Riscos Arquiteturais]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Penetração]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Riscos]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Vulnerabilidades]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Segurança Manuais/Automatizados]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Execução de Testes Fuzz]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Engenharia de Requisitos de Segurança]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Codificação Segura]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Resposta Emergencial ao Código]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Configurações e Gerenciamento de Segurança]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Monitoramento de Eventos de Segurança]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Auditorias de Segurança Externas]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Segurança da Cadeia de Suprimentos de Software]']

In [22]:
frequency_map = {
    'Não aplicável': 0,
    'Anualmente': 1,
    'Mensalmente': 2,
    'Semanalmente': 3,
    'Diariamente': 4,
}

effectiveness_map = {
    'Não Aplicável': 0,
    'Muito Baixo': 1,
    'Baixo': 2,
    'Moderado': 3,
    'Alto': 4,
    'Muito Alto': 5,
}

ease_of_use_map = {
    'Não Aplicável': 0,
    'Muito Baixo': 1,
    'Baixo': 2,
    'Moderado': 3,
    'Alto': 4,
    'Muito Alto': 5,
}

essentiality_map = {
    'Discordo Totalmente': 1,
    'Discordo': 2,
    'Neutro': 3,
    'Concordo': 4,
    'Concordo Totalmente': 5,
}

In [23]:
def generate_description_table(df: pd.DataFrame, order: list = None) -> pd.DataFrame:
    """
    Generates a frequency count table ('Não aplicável', 'Anualmente', etc.)
    for each security activity contained in the DataFrame.

    Parameters:
        df (pd.DataFrame): DataFrame with activity columns and values such as
                           'Diariamente', 'Semanalmente', 'Mensalmente', etc.

    Returns:
        pd.DataFrame: Aggregated table with frequency counts.
    """
    # Copy to avoid modifying the original DataFrame
    description_df = df.copy()

    # Adjust column names if they are in the format "column [x]"
    description_df.columns = [c.split("[")[1].replace("]", "")
                              if "[" in c else c for c in description_df.columns]
    description_df = description_df.rename(
        {"Revisão de Código": "Security Practices"}, axis=1)

    # Create frequency count table
    frequency_description = pd.DataFrame()
    for col in description_df.columns:
        counts = description_df[col].value_counts()
        frequency_description[col] = counts

    # Organize final format (activities as rows)
    frequency_description = frequency_description.fillna(0).astype(int).T

    # Reorder main columns
    frequency_description = frequency_description.reindex(
        columns=order, fill_value=0)

    return frequency_description


frequency_description = generate_description_table(
    df[frequency_cols], order=frequency_map.keys())
ease_of_use_description = generate_description_table(
    df[ease_of_use_cols], order=ease_of_use_map.keys())
effectiveness_description = generate_description_table(
    df[effectiveness_cols], order=effectiveness_map.keys())


In [24]:
for column in frequency_cols:
    df[column] = df[column].map(frequency_map)

for column in effectiveness_cols:
    df[column] = df[column].map(effectiveness_map)

for column in ease_of_use_cols:
    df[column] = df[column].map(ease_of_use_map)

for column in essentiality_cols:
    df[column] = df[column].map(essentiality_map)

In [25]:
experience_col = 'Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'
dev_experience_years_col = 'Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?'
security_experience_years_col = 'Quantos anos de experiência você tem em funções relacionadas a Segurança de Software?'
education_level_col = 'Qual é o seu nível de educação formal concluída?'
seniority_col = 'Por favor, indique a senioridade da posição que ocupa.'
specialist_role_col = 'Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?'
certification_col = 'Você possui alguma certificação na área de segurança da informação? Se sim, quais são elas?'

has_security_experience_response = 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.'

# Roles explicitly focused on software or information security. DevOps engineer
# alone is not considered evidence of security specialization; participants who
# also selected one of these roles remain specialists (roles are multiple choice)
specialist_roles_responses = [
    "Cryptographer",
    "Penetration tester",
    "Privacy engineer",
    "Security engineer"
]

no_certification_response = ["Não possui nenhuma certificação"]


# Security roles written by participants in the "Outros" (free text) field of Q6
specialist_other_roles = [
    "SOC",
    "Gerente de Segurança da Informação",
    "Application secure",
    "Security Analyst",
    "Appsec",
    "AppSec",
    "Cybersegurança",
    "Security Champion",
    "Arquiteto de Segurança",
    "Information Security Analisty I / IAM",
    "Appsec & DevSecOps",
]


# The role question is multiple choice: answers come as a comma-separated
# string, so each selected role (predefined or written in "Outros") is checked
def is_specialist(answer):
    selected_roles = [role.strip() for role in str(answer).split(',')]
    return int(any(role in specialist_roles_responses or role in specialist_other_roles
                   for role in selected_roles))


In [26]:
df_direct_experience = df[df[experience_col] == has_security_experience_response]
df_indirect_experience = df[df[experience_col] != has_security_experience_response]

In [27]:
education_level_map = {
    'Ensino Médio': 1,
    'Ensino Superior': 2,
    'Especialização': 3,
    'Estudante de Mestrado': 4,
    'Mestrado': 5,
    'Estudante de Doutorado': 6,
    'Doutorado': 7
}

seniority_map = {
    'Estagiário': 1,
    'Júnior (até 5 anos)': 2,
    'Pleno (6 a 9 anos)': 3,
    'Sênior (10+ anos)': 4
}

dev_experience_years_map = {
    'Menos de 1 ano': 1,
    'Entre 1 e 3 anos': 2,
    'Entre 4 e 5 anos': 3,
    'Entre 7 e 14 anos': 4,
    'Mais de 15 anos': 5
}

security_experience_years_map = {
    'Nenhuma': 0,
    'Menos de 1 ano': 1,
    'Entre 1 e 3 anos': 2,
    'Entre 4 e 6 anos': 3,
    'Entre 7 e 14 anos': 4,
    'Mais de 15 anos': 5
}


def direct_experience_specialist_interaction(row):
    if row['specialist'] and row['direct_experience']:
        return 3
    elif row['direct_experience'] and not row['specialist']:
        return 2
    else:
        return 1


def direct_experience_specialist_relation(row):
    return (row['specialist'] + row['direct_experience']) / 2


df[dev_experience_years_col] = df[dev_experience_years_col].map(
    dev_experience_years_map)

df[seniority_col] = df[seniority_col].map(seniority_map)

df[security_experience_years_col] = df[security_experience_years_col].map(
    security_experience_years_map)


In [28]:
df[specialist_role_col]

1                 Penetration tester, Security engineer
2              Developer (Backend, frontend, fullstack)
3                                       DevOps engineer
4     Penetration tester, Requirements Engineer, Sec...
5                                     Security engineer
                            ...                        
89    Data Scientist, Developer (Backend, frontend, ...
90                                      DevOps engineer
91                                    Security engineer
92                Penetration tester, Security engineer
93             Developer (Backend, frontend, fullstack)
Name: Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?, Length: 83, dtype: str

In [29]:
# Q6 roles written in "Outros" (Other, free text) that are related to security.
# The predefined options come from the survey script (study_artifacts/Script_survey_pt_br.pdf)
q6_options = [
    "Cryptographer", "Data Scientist", "Developer (Backend, frontend, fullstack)",
    "Penetration tester", "DevOps engineer", "Privacy engineer", "Product owner",
    "Project Lead / Project Manager", "Requirements Engineer", "Security engineer",
    "Solution Architect", "Test Manager / Tester", "UX/UI designer",
]

# Terms that identify a security-related role in the free text
security_terms = r'secur|seguran|appsec|cyber|devsecops|\bsoc\b'


def other_roles(answer):
    """Returns the free-text roles of an answer, removing the predefined options."""
    answer = str(answer)
    for option in q6_options:
        # Options are removed as whole strings first, because some contain commas
        answer = answer.replace(option, '')
    return [role.strip() for role in answer.split(',') if role.strip()]


other_roles_df = pd.DataFrame({
    'Q6 answer': df[specialist_role_col],
    'Other roles': df[specialist_role_col].apply(other_roles),
    'Specialist (current rule)': df[specialist_role_col].apply(is_specialist).astype(bool),
})
other_roles_df['Security role in Other'] = other_roles_df['Other roles'].apply(
    lambda roles: [role for role in roles if pd.Series(role).str.contains(security_terms, case=False).iloc[0]]
)

security_others = other_roles_df[other_roles_df['Security role in Other'].str.len() > 0]

print('Participants who wrote any role in "Outros" (Other):', int((other_roles_df['Other roles'].str.len() > 0).sum()))
print('Participants with a security role in "Outros" (Other):', len(security_others))
print('  - already specialists by the current rule:', int(security_others['Specialist (current rule)'].sum()))
print('  - would be added if these roles were included:', int((~security_others['Specialist (current rule)']).sum()))

display(security_others[['Security role in Other', 'Specialist (current rule)', 'Q6 answer']])

Participants who wrote any role in "Outros" (Other): 14
Participants with a security role in "Outros" (Other): 11
  - already specialists by the current rule: 11
  - would be added if these roles were included: 0


,Security role in Other,Specialist (current rule),Q6 answer
16,[SOC],True,SOC
17,[Gerente de Segurança da Informação],True,Gerente de Segurança da Informação
18,[Application secure],True,Application secure
20,[Security Analyst],True,Security Analyst
23,[Appsec],True,Appsec
35,[Cybersegurança],True,Cybersegurança
42,[Security Champion],True,Security Champion
55,[Arquiteto de Segurança],True,Arquiteto de Segurança
69,[Information Security Analisty I / IAM],True,Information Security Analisty I / IAM
72,[AppSec],True,"Project Lead / Project Manager, Security engin..."


In [30]:
df['specialist'] = df[specialist_role_col].apply(is_specialist)

df['direct_experience'] = df[experience_col].apply(
    lambda x: 1 if x == has_security_experience_response else 0)

df['certification'] = df[certification_col].apply(
    lambda x: 0 if x in no_certification_response else 1)

df['Qual é o seu nível de educação formal concluída?'] = df['Qual é o seu nível de educação formal concluída?'].map(
    education_level_map)

df['direct_experience_specialist_interaction'] = df.apply(
    direct_experience_specialist_interaction, axis=1)

df['direct_experience_specialist_relation'] = df.apply(
    direct_experience_specialist_relation, axis=1)

## Create the Spreadsheet

In [31]:
import re

# Definition of direct, indirect, large, and small responses
direct_response = 1
indirect_response = 0

small_company_response = [
    "Até 9 funcionários",
    "De 10 a 49 funcionários",
    "De 50 a 99 funcionários"
]
large_company_response = [
    "De 100 a 499 funcionários",
    "De 500 a 999 funcionários",
    "Mais de 1000 funcionários"
]

specialist_response = 1
non_specialist_response = 0

# Correct column names
direct_experience_col = 'direct_experience'
company_size_col = 'Qual o tamanho da empresa para a qual você trabalha atualmente?'
specialist_col = 'specialist'
systems_col = 'Quais são os tipos de sistemas que você trabalha?'

# Filter direct and indirect responses
direct_responses = df[df[direct_experience_col] == direct_response].copy()
indirect_responses = df[df[direct_experience_col] == indirect_response].copy()

# Filter large and small company responses
large_companies = df[df[company_size_col].isin(large_company_response)].copy()
small_companies = df[df[company_size_col].isin(small_company_response)].copy()

specialists = df[df[specialist_col] == specialist_response].copy()
non_specialists = df[df[specialist_col] == non_specialist_response].copy()


In [32]:
print("Number of participants in the Direct Experience group:", len(direct_responses))
print("Number of participants in the Indirect Experience group:", len(indirect_responses))
print("--" * 35)

print("Number of participants in the Big group:", len(large_companies))
print("Number of participants in the Small group:", len(small_companies))
print("--" * 35)

print("Number of participants in the Specialists group:", len(specialists))
print("Number of participants in the Non-specialists group:",
      len(non_specialists))
print("--" * 35)

print(f"Percentage of participants in the Big group: {len(large_companies)/88:.1f}")
print(f"Percentage of participants in the Small group: {len(small_companies)/88:.1f}")


Number of participants in the Direct Experience group: 62
Number of participants in the Indirect Experience group: 21
----------------------------------------------------------------------
Number of participants in the Big group: 65
Number of participants in the Small group: 16
----------------------------------------------------------------------
Number of participants in the Specialists group: 57
Number of participants in the Non-specialists group: 26
----------------------------------------------------------------------
Percentage of participants in the Big group: 0.7
Percentage of participants in the Small group: 0.2


In [33]:
results = []

# Function that calculates means, p-value, effect size and returns the results for a pair of groups
def calculate_data(group1, group2, column):
    group1 = group1[column][group1[column] != 0]
    group2 = group2[column][group2[column] != 0]
    group1_score = group1.mean()
    group2_score = group2.mean()

    stat, p_value = stats.mannwhitneyu(
        group1.dropna(), group2.dropna(), alternative='two-sided')

    # cliffs delta
    effect_size = cliffs_delta(group1, group2)

    return group1_score, group2_score, p_value, effect_size


# Function that calculates the distribution of responses on the Likert scale
# and the weighted mean for each column
def calculate_likert_distribution(group, column):

    # 'Não Aplicável' (Not Applicable, coded as 0) is excluded, as in the group means
    mapped_group = group[column][group[column] != 0]
    total_responses = mapped_group.value_counts().sort_index()
    weighted_mean = sum(
        total_responses.index *
        (total_responses / total_responses.sum())
    )
    return total_responses.to_dict(), weighted_mean


In [34]:
def detect_practice_and_attribute(text: str):
    """
    Detects the practice (text inside brackets) and the attribute of a survey
    column. The survey text is matched in Portuguese ('facilidade', 'efetividade',
    'frequência', 'essencial'), as it appears in the data.

    Returns:
        - 'practice': English name of the practice (or None if there are no brackets)
        - 'attribute': 'Ease of use', 'Effectiveness', 'Frequency of use',
          'Essentiality' or None
    """
    # --- 1. Detect practice (text inside brackets) ---
    match = re.search(r'\[(.*?)\]', text)
    practice = practice_name_en[match.group(1).strip()] if match else None

    # --- 2. Detect attribute ---
    attribute = None
    if 'facilidade' in text.lower():
        attribute = 'Ease of use'
    elif 'efetividade' in text.lower():
        attribute = 'Effectiveness'
    elif 'frequência' in text.lower():
        attribute = 'Frequency of use'
    elif 'essencial' in text.lower():
        attribute = 'Essentiality'

    return practice, attribute


In [35]:
# Loop that iterates over each survey column, performs the analyses,
# and stores the results
for survey_col in security_practices_survey_cols:
    practice, attribute = detect_practice_and_attribute(survey_col)

    total_responses, weighted_mean = calculate_likert_distribution(df, survey_col)

    direct_score, indirect_score, p_value_direct_indirect, effect_size_direct_indirect = calculate_data(
        direct_responses, indirect_responses, survey_col
    )
    
    big_score, small_score, p_value_big_small, effect_size_big_small = calculate_data(
        large_companies, small_companies, survey_col
    )
    
    specialist_score, non_specialist_score, p_value_specialist_non_specialist, effect_size_specialist_non_specialist = calculate_data(
        specialists, non_specialists, survey_col
    )

    # Create a dictionary with all computed results and append it to the results list
    row = {
        'Practice': practice,
        'Attribute': attribute,
        'Overall Mean': weighted_mean,
        'Mean Direct': direct_score,
        'Mean Indirect': indirect_score,
        'P-value Direct/Indirect': p_value_direct_indirect,
        'Effect Size Direct/Indirect': effect_size_direct_indirect,

        'Mean Large': big_score,
        'Mean Small': small_score,
        'P-value Big/Small': p_value_big_small,
        'Effect Size Big/Small': effect_size_big_small,

        'Mean Specialist': specialist_score,
        'Mean Non-Specialist': non_specialist_score,
        'P-value Specialist/Non-Specialist': p_value_specialist_non_specialist,
        'Effect Size Specialist/Non-Specialist': effect_size_specialist_non_specialist,
    }

    results.append(row)


In [36]:
results

[{'Practice': 'Code Review',
  'Attribute': 'Ease of use',
  'Overall Mean': 3.5555555555555554,
  'Mean Direct': np.float64(3.5964912280701755),
  'Mean Indirect': np.float64(3.4),
  'P-value Direct/Indirect': np.float64(0.7452926126336895),
  'Effect Size Direct/Indirect': (0.05380116959064327, 'negligible'),
  'Mean Large': np.float64(3.618181818181818),
  'Mean Small': np.float64(3.533333333333333),
  'P-value Big/Small': np.float64(0.6700499888775223),
  'Effect Size Big/Small': (0.0703030303030303, 'negligible'),
  'Mean Specialist': np.float64(3.5),
  'Mean Non-Specialist': np.float64(3.6666666666666665),
  'P-value Specialist/Non-Specialist': np.float64(0.521800834942814),
  'Effect Size Specialist/Non-Specialist': (-0.09027777777777778,
   'negligible')},
 {'Practice': 'Threat Modeling',
  'Attribute': 'Ease of use',
  'Overall Mean': 3.238095238095238,
  'Mean Direct': np.float64(3.2830188679245285),
  'Mean Indirect': np.float64(3.0),
  'P-value Direct/Indirect': np.float64(

## Holm-Bonferroni correction

Each group comparison (Direct/Indirect, Big/Small, Specialist/Non-Specialist) runs one Mann-Whitney U test per practice and attribute, so the p-values are adjusted for multiple comparisons with the Holm-Bonferroni method.

The family of tests is defined by `HOLM_FAMILY_COLUMNS`: the correction is applied separately to each comparison within each group of rows sharing the same values in these columns.

- `['Attribute']` (default): one family per comparison and attribute (16 tests, one per practice).
- `[]`: one family per comparison (64 tests, all practices and attributes).

In [37]:
# Columns that define each family of tests (see the markdown cell above)
HOLM_FAMILY_COLUMNS = ['Attribute']


def holm_bonferroni(p_values: pd.Series) -> pd.Series:
    """
    Returns the Holm-Bonferroni adjusted p-values, ignoring missing values.
    """
    valid = p_values.dropna().sort_values()
    m = len(valid)

    # Step-down adjustment: p_(i) * (m - i + 1), forced to be monotonic and capped at 1
    adjusted = (valid * np.arange(m, 0, -1)).cummax().clip(upper=1)

    return adjusted.reindex(p_values.index)


def apply_holm_correction(analysis_df: pd.DataFrame, family_cols: list) -> pd.DataFrame:
    """
    Adds a 'P-value Holm <comparison>' column right after each 'P-value <comparison>' column.
    """
    analysis_df = analysis_df.copy()
    p_value_cols = [col for col in analysis_df.columns if col.startswith('P-value ')]

    for p_col in p_value_cols:
        holm_col = p_col.replace('P-value ', 'P-value Holm ')

        if family_cols:
            adjusted = analysis_df.groupby(family_cols)[p_col].transform(holm_bonferroni)
        else:
            adjusted = holm_bonferroni(analysis_df[p_col])

        analysis_df.insert(analysis_df.columns.get_loc(p_col) + 1, holm_col, adjusted)

    return analysis_df

In [38]:
analysis_results_path = Path('results/RQ2')
analysis_results_path.mkdir(parents=True, exist_ok=True)

analysis_df = pd.DataFrame(results)
analysis_df = apply_holm_correction(analysis_df, HOLM_FAMILY_COLUMNS)

analysis_df.to_excel(
    analysis_results_path / 'analysis_results_raw.xlsx',
    index=False
)


## Formatting table

In [39]:
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
import os

# --- Paths ---
input_path = results_path / "analysis_results_raw.xlsx"
output_path = results_path / "analysis_results_formatted.xlsx"

df = pd.read_excel(input_path)

print("Columns detected in the input file:", list(df.columns))

writer = pd.ExcelWriter(output_path, engine='openpyxl')
pd.DataFrame().to_excel(writer, sheet_name="Analysis", index=False)
writer.close()

workbook = openpyxl.load_workbook(output_path)
worksheet = workbook["Analysis"]

main_headers = [
    'Practice', 'Attribute', 'Overall Mean',
    'Direct vs. Indirect',
    'Large vs. Small',
    'Specialist vs. Non-specialist'
]

# Subcolumns: standard 5 subcolumns per comparison (Mean, Mean, p-value, Holm p-value, d)
# Final column organization (total 18 columns):
headers = [
    'Practice',
    'Attribute',
    'Overall Mean',

    'Mean Direct',
    'Mean Indirect',
    'P-value Direct/Indirect',
    'P-value Holm Direct/Indirect',
    'Effect Size Direct/Indirect',

    'Mean Large',
    'Mean Small',
    'P-value Big/Small',
    'P-value Holm Big/Small',
    'Effect Size Big/Small',

    'Mean Specialist',
    'Mean Non-Specialist',
    'P-value Specialist/Non-Specialist',
    'P-value Holm Specialist/Non-Specialist',
    'Effect Size Specialist/Non-Specialist'
]

merge_cells = [
    (1, 1, 1, 2),   # Practice
    (2, 2, 1, 2),   # Attribute
    (3, 3, 1, 2),   # Overall Mean
    (4, 8, 1, 1),   # Direct vs Indirect
    (9, 13, 1, 1),  # Large vs Small
    (14, 18, 1, 1)  # Specialist vs Non-specialist
]

# --- Styles ---
header_font = Font(bold=True)
header_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
center_alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
thin_border = Border(
    left=Side(style="thin"),
    right=Side(style="thin"),
    top=Side(style="thin"),
    bottom=Side(style="thin")
)

# --- Build headers in rows 1 and 2 ---
for start_col, end_col, start_row, end_row in merge_cells:
    worksheet.merge_cells(
        start_row=start_row,
        start_column=start_col,
        end_row=end_row,
        end_column=end_col
    )

# Insert row 1 headers
column_index = 1
for header in main_headers:
    cell = worksheet.cell(row=1, column=column_index, value=header)
    cell.alignment = center_alignment
    cell.font = header_font
    cell.fill = header_fill

    if header == 'Overall Mean':
        column_index += 1
    elif header in (
        'Direct vs. Indirect',
        'Large vs. Small',
        'Specialist vs. Non-specialist'
    ):
        column_index += 5
    else:
        column_index += 1

# Insert row 2 headers (subcolumns)
# Column 3 (Overall Mean) is vertically merged, so subheaders start at column 4
group_sub_headers = ['Mean', 'Mean', 'p-value', 'p-value (Holm)', 'd']
full_sub_headers = group_sub_headers * 3
current_column = 4

for sub_header in full_sub_headers:
    cell = worksheet.cell(row=2, column=current_column, value=sub_header)
    cell.alignment = center_alignment
    cell.font = Font(bold=True)
    cell.fill = header_fill
    current_column += 1

# Apply border and alignment to header rows
for row in worksheet.iter_rows(min_row=1, max_row=2, min_col=1, max_col=18):
    for cell in row:
        cell.border = thin_border
        cell.alignment = center_alignment

# --- Write body ---
current_row = 3
for practice_name, group in df.groupby("Practice"):
    num_rows = len(group)
    start_row = current_row
    end_row = current_row + num_rows - 1

    worksheet.merge_cells(
        start_row=start_row,
        start_column=1,
        end_row=end_row,
        end_column=1
    )

    practice_cell = worksheet.cell(row=start_row, column=1, value=practice_name)
    practice_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    practice_cell.font = Font(bold=True)
    practice_cell.border = thin_border

    for i, (_, row) in enumerate(group.iterrows()):
        row_index = start_row + i

        for col, header in enumerate(headers[1:], start=2):
            value = row.get(header)

            if value is None or (isinstance(value, float) and pd.isna(value)):
                display_value = "-"
            elif isinstance(value, str) and '(' in value:
                value = eval(value)

                if value[0] > 0:
                    display_value = '(+)'
                elif value[0] < 0:
                    display_value = '(-)'
                elif value[0] == 0:
                    display_value = '(+)'

                if value[1] == 'large':
                    display_value += '***'
                elif value[1] == 'medium':
                    display_value += '**'
                elif value[1] == 'small':
                    display_value += '*'
                elif value[1] == 'negligible':
                    pass

            elif isinstance(value, (int, float)) and header.startswith('P-value'):
                display_value = f"{value:.3f}"
            elif isinstance(value, float):
                display_value = f"{value:.2f}"
            else:
                display_value = value

            cell = worksheet.cell(row=row_index, column=col, value=display_value)
            cell.alignment = center_alignment
            cell.border = thin_border

    current_row = end_row + 1

# Auto-adjust column widths
for col in range(1, 19):
    col_letter = get_column_letter(col)
    width = 10 if col > 2 else 30
    worksheet.column_dimensions[col_letter].width = width

# Save file
workbook.save(output_path)
print(f"File formatted with a two-line header and saved at: {output_path}")


Columns detected in the input file: ['Practice', 'Attribute', 'Overall Mean', 'Mean Direct', 'Mean Indirect', 'P-value Direct/Indirect', 'P-value Holm Direct/Indirect', 'Effect Size Direct/Indirect', 'Mean Large', 'Mean Small', 'P-value Big/Small', 'P-value Holm Big/Small', 'Effect Size Big/Small', 'Mean Specialist', 'Mean Non-Specialist', 'P-value Specialist/Non-Specialist', 'P-value Holm Specialist/Non-Specialist', 'Effect Size Specialist/Non-Specialist']
File formatted with a two-line header and saved at: results/RQ2/analysis_results_formatted.xlsx


In [40]:
from openpyxl import Workbook, load_workbook
from openpyxl.styles import PatternFill, Font

# 1. Color and Style Configuration

# Color when Mean 1 > Mean 2 (Dark Gray)
GRAY_DARK_FILL = PatternFill(
    start_color="A9A9A9",
    end_color="A9A9A9",
    fill_type="solid"
)

# Color when Mean 2 > Mean 1 (Light Gray)
GRAY_LIGHT_FILL = PatternFill(
    start_color="D3D3D3",
    end_color="D3D3D3",
    fill_type="solid"
)

# Font for bold p-value
BOLD_FONT = Font(bold=True)

# 3. Column blocks to be analyzed (1-based columns, 1 = A)
# Structure: [Mean 1, Mean 2, Holm p-value, d]
# Significance is based on the Holm-Bonferroni adjusted p-value
col_blocks = [
    [4, 5, 7, 8],     # Direct vs. Indirect (D, E, G, H)
    [9, 10, 12, 13],  # Large vs. Small (I, J, L, M)
    [14, 15, 17, 18]  # Specialist vs. Non-specialist (N, O, Q, R)
]

# 5. Function to apply formatting
def apply_formatting(xlsx_path, col_blocks):
    # Load workbook and active worksheet
    wb = load_workbook(xlsx_path)
    ws = wb.active

    # Iterate over rows, starting from row 3 (where data for the first practice begins)
    # ws.max_row is the index of the last row with data
    for row_idx in range(3, ws.max_row + 1):
        for col_block in col_blocks:
            col_mean1, col_mean2, col_p_value, col_d = col_block

            # Try to retrieve cell values.
            # Error handling is important if there are empty or non-numeric cells.
            try:
                # Getting cell values
                p_value_cell = ws.cell(row=row_idx, column=col_p_value)
                mean1_cell = ws.cell(row=row_idx, column=col_mean1)
                mean2_cell = ws.cell(row=row_idx, column=col_mean2)
                d_cell = ws.cell(row=row_idx, column=col_d)

                # Safely converting values to float.
                # If the value is None or not convertible, skip
                p_value = float(p_value_cell.value)
                mean1 = float(mean1_cell.value)
                mean2 = float(mean2_cell.value)

            except (ValueError, TypeError):
                # Ignore this row/block if values cannot be converted
                continue

            # **Formatting Logic**

            # If the Holm adjusted p-value is less than or equal to 0.05 (Significance)
            if p_value <= 0.05:

                # 1. If Mean 1 is greater than Mean 2
                if mean1 > mean2:
                    # Apply bold to Holm p-value
                    p_value_cell.font = BOLD_FONT
                    # Fill 'd' with dark gray
                    d_cell.fill = GRAY_DARK_FILL

                # 2. If Mean 2 is greater than Mean 1
                elif mean2 > mean1:
                    # Apply bold to Holm p-value
                    p_value_cell.font = BOLD_FONT
                    # Fill 'd' with light gray
                    d_cell.fill = GRAY_LIGHT_FILL

                # Note: If mean1 == mean2, no bold or fill is applied to 'd',
                # but it still passes the p_value <= 0.05 condition.

    # Save the modified file
    wb.save(xlsx_path)


print(f"Applying formatting rules to '{output_path}'...")
apply_formatting(output_path, col_blocks)

print(f"✅ Formatting completed! The file '{output_path}' was created/updated.")


# Visual example of the rule (for reference):


Applying formatting rules to 'results/RQ2/analysis_results_formatted.xlsx'...
✅ Formatting completed! The file 'results/RQ2/analysis_results_formatted.xlsx' was created/updated.
